In [1]:
import numpy as np
import pandas as pd
import subprocess
import time

In [2]:
# presets, functions

# preset values for the analysis:
#____________________________________________________________________________________________________________________
# physical constants
Z = 6
A = 12
mass_nucleon = 0.938273
mass_nucleus = A * 0.931494
alpha_fine = 1 / 137.036
Ex_cut = 0.03
Ex_cut_lowq = 0.1

# three-momentum bin centers
qvcenters = [0.100, 0.148, 0.167, 0.205, 0.240, 0.300, 0.380, 0.475, 0.570, 0.649, 0.756, 0.991, 1.619, 1.921, 2.213, 2.500, 2.783, 3.500]
# three-momentum edges
qvbins = [0.063, 0.124, 0.158, 0.186, 0.223, 0.270, 0.340, 0.428, 0.523, 0.609, 0.702, 0.878, 1.302, 1.770, 2.067, 2.357, 2.642, 2.923, 4.500]
# four-momentum squared bin names, in string format
qvbin_names = ['[0.063,0.124]', '[0.124,0.158]', '[0.158,0.186]', '[0.186,0.223]', '[0.223,0.270]', '[0.270,0.340]', '[0.340,0.428]', '[0.428,0.523]', '[0.523,0.609]',
                '[0.609,0.702]', '[0.702,0.878]', '[0.878,1.302]', '[1.302,1.770]', '[1.770,2.067]', '[2.067,2.357]', '[2.357,2.642]', '[2.642,2.923]', '[2.923,4.500]']

# four-momentum squared bin centers
Q2centers = [0.010, 0.020, 0.026, 0.040, 0.056, 0.093, 0.120, 0.160, 0.265, 0.380, 0.500, 0.800, 1.250, 1.750, 2.250, 2.750, 3.250, 3.750]
# four-momentum squared bin edges
Q2bins = [0.004, 0.015, 0.025, 0.035, 0.045, 0.070, 0.100, 0.145, 0.206, 0.322, 0.438, 0.650, 1.050, 1.500, 2.000, 2.500, 3.000, 3.500, 4.000]
# four-momentum squared bin names, in string format
Q2bin_names = ['[0.004,0.015]', '[0.015,0.025]', '[0.025,0.035]', '[0.035,0.045]', '[0.045,0.070]', '[0.070,0.100]', '[0.100,0.145]', '[0.145,0.206]', '[0.206,0.322]',
                '[0.322,0.438]', '[0.438,0.650]', '[0.650,1.050]', '[1.050,1.500]', '[1.500,2.000]', '[2.000,2.500]', '[2.500,3.000]', '[3.000,3.500]', '[3.500,4.000]']

# # New Version
# dataSet_to_normalization = {1: 0.95971, 2: 0.96416, 3: 1.0744, 4: 0.99482, 5: 0.93381, 6: 1.0126, 
#                             7: 0.96716, 8: 1.0238, 9: 0.97904, 10: 0.99064, 11: 0.98384, 12: 1.0000, 
#                             13: 1.0163, 14: 1.0300, 15: 1.0190, 16: 0.95853, 17: 1.0174, 18: 1.0168, 
#                             19: 1.0794, 20: 1.0000, 21: 0.9500, 22: 1.1095, 23: 0.9310, 24: 1.0019, 
#                             25: 0.8500, 26: 1.0000, 33: 0.9980, 34: 0.9677, 35: 0.9561}

# dataSet_to_normError = {1: 0.62926E-02, 2: 0.12908E-01, 3: 0.80983E-02, 4: 0.69809E-02, 5: 0.16758E-01, 6: 0.92261E-02, 
#                         7: 0.15546E-01, 8: 0.65203E-02, 9: 0.55606E-02, 10: 0.75245E-02, 11: 0.25318E-01, 12: 0.0, 
#                         13: 0.17632E-02, 14: 0.91993E-02, 15: 0.63181E-02, 16: 0.25582E-01, 17: 0.42184E-01, 18: 0.68067E-01, 
#                         19: 0.35847E-01, 20: 0.0, 21: 0.25, 22: 0.1, 23: 0.1, 24: 0.184E-01, 
#                         25: 0.02, 26: 0.02, 33: 0.415E-01, 34: 0.173E-01, 35: 0.231E-01}

dataSet_to_normalization = {1: 0.9919, 2: 0.9787, 3: 1.06, 4: 0.9924, 5: 0.9878,
                            6: 1.0108, 7: 0.9743, 8: 1.0071, 9: 0.9888, 10: 0.9934,
                            11: 1.0149, 12: 0.9981, 13: 1.0029, 14: 1.0125, 15: 1.0046,
                            16: 1.0019, 17: 1.10, 18: 1.000, 19: 1.150, 20: 1.0,
                            21: 0.95, 22: 1.100, 23: 0.9, 24: 1.03, 25: 0.85}

dataSet_to_normError = {1: 0.0024, 2: 0.0086, 3: 0.1000, 4: 0.0046, 5: 0.0083,
                        6: 0.0053, 7: 0.0133, 8: 0.0033, 9: 0.0034, 10: 0.0051,
                        11: 0.0153, 12: 0.0067, 13: 0.0070, 14: 0.0149, 15: 0.0031,
                        16: 0.0029, 17: 0.0130, 18: 0.2000, 19: 0.2300, 20: 0.0,
                        21: 0.25, 22: 0.1000, 23: 0.1000, 24: 0.02, 25: 0.02}

In [3]:
def pauli_blocking(ex):
    if ex <= 20:
        return np.exp(-73.3 / ex)
    elif 20 < ex <= 140: 
        return 8.3714e-2 - 9.8343e-3 * ex + 4.1222e-4 * ex**2 - 3.4762e-6 * ex**3 + 9.3537e-9 * ex**4
    else:
        return np.exp(-24.2/ex)
        
def quasi_deuteron(ex):
    N = A - Z
    ex = ex * 1e3
    if ex < 2.224:
        return 0
    else:    
        sigma = 397.8 * (N * Z / A) * ((ex - 2.224)**(3 / 2)) * (ex**-3) * pauli_blocking(ex)
        return sigma * 0.1 * 0.1975**-2

def dipole_E(Q2):
    return 1 / ((1 + Q2 / 0.5)**5)

def QD_suppression(nu, center = 0.12, width = 0.005):
    return 1 / (np.exp((nu - center) / width) + 1)

def RT_quasi_deuteron(nus = [], q2s = [], exs = []):
    QDs = []            
    for ex in exs:
        QDs.append(quasi_deuteron(ex))
    QDs = np.array(QDs)
    GEs = dipole_E(q2s)
    GEs = np.array(GEs)
    RTQD = GEs**2 * QDs * nus / (2 * (np.pi**2) * alpha_fine)
    RTQD = RTQD * QD_suppression(nu = nus)
    return RTQD * 1e-3

In [4]:
# read dataframe from csv file
def prepare_df(df):

    # calculate normalized cross section:
    if 'error' not in df.columns:
        df['error'] = df['cross'] * 0.02
    if 'dataSet' not in df.columns:
        df['dataSet'] = -1
        df['normalization'] = 1.0
        df['normError'] = 0.0
    else:
        df["normalization"] = df["dataSet"].map(dataSet_to_normalization)
        df["normError"] = df["dataSet"].map(dataSet_to_normError)
    df['system_err'] = 0.0
    df['normCross'] = df['cross'] * df['normalization']
    df['error'] = np.sqrt(df['error']**2 + ((df['system_err'] * df['cross'])**2))
    df['normCrossError'] = df['normCross'] * np.sqrt((df['error'] / df['cross'])**2 + (df['normError'] / df['normalization'])**2)
    print(df.loc[df['normalization'] == 1, 'dataSet'].unique())
    
    # calculate the kinematic variables:
    df["Veff"] = 0.0031
    df["ThetaRad"] = df["ThetaDeg"] * np.pi / 180
    df["sin2(T/2)"] = (np.sin(df["ThetaRad"] / 2))**2
    df["cos2(T/2)"] = (np.cos(df["ThetaRad"] / 2))**2
    df["tan2(T/2)"] = (np.tan(df["ThetaRad"] / 2))**2
    df["Ex"] = df["nu"] - (df["E0"] - df["E0"] / (1 + 2 * df["E0"] * df["sin2(T/2)"] / mass_nucleus))
    df["W2original"] = mass_nucleon**2 + 2 * mass_nucleon * df["nu"] - (4 * df["E0"] * (df["E0"] - df["nu"]) * df["sin2(T/2)"])
    df["Ffoc2"] = ((df["E0"] + df["Veff"]) / df["E0"])**2
    df["Ep"] = df["E0"] - df["nu"]
    # ______________________starting here: effective values only______________________
    df["E0original"] = df["E0"]
    df["E0"] = df["E0"] + df["Veff"]
    df["Ep"] = df["Ep"] + df["Veff"]
    df["R"] = 1.1 * (df["A"])**(1/3) + 0.86 / ((df["A"])**(1/3))
    df["Q2"] = 4 * df["E0"] * df["Ep"] * df["sin2(T/2)"]
    df["qv2"] = df["nu"]**2 + df["Q2"]
    df["qv"] = np.sqrt(df["qv2"])
    df["W2"] = mass_nucleon**2 + 2 * mass_nucleon * df["nu"] - df["Q2"]
    df["epsilon"] = 1 / (1 + 2 * (1 + (df["nu"]**2) / df["Q2"]) * df["tan2(T/2)"])
    df["gamma"] = alpha_fine * df["Ep"] * (df["W2"] - mass_nucleon**2) / (( 4 * ((np.pi)**2) * df["Q2"] * mass_nucleon * df["E0"]) * (1 - df["epsilon"]))
    df["Sig_R"] = df["normCross"] / df["gamma"]
    df["D_sig_R"] = df["error"] / df["gamma"]
    df["Sig_mott"] = df["Ffoc2"] * alpha_fine**2 * df["cos2(T/2)"] * (2 * df["E0"] * df["sin2(T/2)"])**-2

    # calculate the Rosenbluth quantity:
    df["Hcc"] = ((df["qv"]**4) / (4 * (alpha_fine**2) * (df["Ep"]**2) * (df["cos2(T/2)"] + 2 * (df["qv2"] / df["Q2"]) * df["sin2(T/2)"]))) / df["Ffoc2"]
    df["Hcc_Sig(nb)"] = df["Hcc"] * df["normCross"]
    df["Hcc_error(nb)"] = df["Hcc"] * df["normCrossError"]
    df["Hcc_Sig(GeV)"] = df["Hcc_Sig(nb)"] / ((0.1973269**2) * 10000000)
    df["Hcc_error(GeV)"] = df["Hcc_error(nb)"] / ((0.1973269**2) * 10000000)
    
    # RT quasi deuteron added 2025 July 18
    df["RT_QD_data"] = RT_quasi_deuteron(nus = df['nu'], q2s = df['Q2'], exs = df['Ex'])

    # subdivide the data into bins
    df['qvbin'] = 0
    df['qvcenter'] = 0
    df["qvbin"] = pd.cut(x=df["qv"], bins = qvbins, labels = qvbin_names, right=True)
    df["qvcenter"] = pd.cut(x=df["qv"], bins = qvbins, labels = qvcenters, right=True)
    df['qvcenter'] = pd.to_numeric(df['qvcenter'])
    df['Q2bin'] = 0
    df['Q2center'] = 0
    df["Q2bin"] = pd.cut(x = df["Q2"], bins = Q2bins, labels = Q2bin_names, right = True)
    df["Q2center"] = pd.cut(x = df["Q2"], bins = Q2bins, labels = Q2centers, right = True)
    df['Q2center'] = pd.to_numeric(df['Q2center'])
    df = df.dropna()

    # bin-centering related:
    df['Exbin_qv'] = 0.0
    df['Excenter_qv'] = 0.0
    df['nucenter_ex_qv'] = 0.0
    df['epcenter_ex_qv'] = 0.0

    df['W2bin_qv'] = 0.0
    df['W2center_qv'] = 0.0
    df['nucenter_w2_qv'] = 0.0
    df['epcenter_w2_qv'] = 0.0

    df['Exbin_q2'] = 0.0
    df['Excenter_q2'] = 0.0
    df['nucenter_ex_q2'] = 0.0
    df['epcenter_ex_q2'] = 0.0

    df['W2bin_q2'] = 0.0
    df['W2center_q2'] = 0.0
    df['nucenter_w2_q2'] = 0.0
    df['epcenter_w2_q2'] = 0.0

    def Exedges_epsilon_range(df = None, edges = None, min_range = 0.25):
        i = 0
        while i < len(edges) - 2:
            lo, hi = edges[i], edges[i + 2]
            # Calculate y-range in the merged bin [lo, hi)
            sub = df[(df['Ex'] >= lo) & (df['Ex'] < hi)]
            if not sub.empty and (sub['epsilon'].max() - sub['epsilon'].min()) < min_range:
                # Merge by removing mid-edge
                edges = np.delete(edges, i + 1)
                # Step back to re-check previous merge
                if i > 0: i -= 1
            else:
                i += 1
        return edges

    def W2edges_epsilon_range(df = None, edges = None, min_range = 0.25):
        i = 0
        while i < len(edges) - 2:
            lo, hi = edges[i], edges[i + 2]
            # Calculate y-range in the merged bin [lo, hi)
            sub = df[(df['W2'] >= lo) & (df['W2'] < hi)]
            if not sub.empty and (sub['epsilon'].max() - sub['epsilon'].min()) < min_range:
                # Merge by removing mid-edge
                edges = np.delete(edges, i + 1)
                # Step back to re-check previous merge
                if i > 0: i -= 1
            else:
                i += 1
        return edges
    
    W2ns = np.array([15,20,15,15,10,
                20,15,12,10,10,
                8,15,15,15,15,
                15,15,15])

    for i in range(len(qvcenters)):
        qvcenter = qvcenters[i]

        # Ex < 30MeV:
        mask = (df['qvcenter'] == qvcenter) & (df['Ex'] < Ex_cut)
        if qvcenter == 0.1:
            mask = (df['qvcenter'] == qvcenter) & (df['Ex'] < Ex_cut_lowq)
        if len(df.loc[mask, 'Ex']) > 0:
            n_bins = max(1, len(df.loc[mask, 'Ex']) // 5)
            Exedges = np.quantile(df.loc[mask, 'Ex'], np.linspace(0, 1, n_bins + 1))
            Exedges = Exedges_epsilon_range(df = df.loc[mask], edges = Exedges)
            Excenters = (Exedges[:-1] + Exedges[1:]) / 2
            df.loc[mask, 'Exbin_qv'] = pd.cut(df.loc[mask, 'Ex'], bins = Exedges, labels = False, include_lowest = True, duplicates = 'drop')
            df.loc[mask, 'Excenter_qv'] = df.loc[mask, 'Exbin_qv'].map(lambda i: Excenters[int(i)] if pd.notnull(i) else np.nan)
            df.loc[mask, 'nucenter_ex_qv'] = np.sqrt(mass_nucleus**2 + qvcenter**2 + 2 * mass_nucleus * df.loc[mask, 'Excenter_qv']) - mass_nucleus
            df.loc[mask, 'epcenter_ex_qv'] = 1 / (1 + 2 * (1 + (df.loc[mask, 'nucenter_ex_qv']**2) / (qvcenter**2 - df.loc[mask, 'nucenter_ex_qv']**2)) * df.loc[mask, 'tan2(T/2)'])

        # Ex >= 30MeV:
        mask = (df['qvcenter'] == qvcenter) & (df['Ex'] >= Ex_cut)
        if qvcenter == 0.1:
            mask = (df['qvcenter'] == qvcenter) & (df['Ex'] >= Ex_cut_lowq)
        if len(df.loc[mask, 'W2']) > 0:
            n_bins = max(1,len(df.loc[mask, 'W2']) // W2ns[i])
            W2edges = np.quantile(df.loc[mask, 'W2'], np.linspace(0, 1, n_bins + 1))
            if qvcenter == 0.3:
                W2edges = np.delete(W2edges,[-2,-3])
            if qvcenter == 0.38:
                W2edges = np.delete(W2edges,[-2,-3,-4,-7,-8])
            if qvcenter == 0.475:
                W2edges = np.delete(W2edges, [-2,-3,-4,-5,-6,-7,-8,-9,-10,-12,-13,-14,-15,-16,-17,-18,-35])
            if qvcenter == 0.57:
                W2edges = np.delete(W2edges, [-5,-6,-7,-8,-9,-10,-11,-12,-13,-14,-15,-16,-17,-18,-20,-28,-30,-33,-40])
            if qvcenter == 0.649:
                W2edges = np.delete(W2edges,[-3,-4,-5,-6,-7,-8,-9,-10,-11,-12,-14,-15,-16])
            if qvcenter == 0.756:
                W2edges = np.delete(W2edges,[-6,-7,-8,-9,-10,-11,-12,-13,-14,-15,-16,-17,-18,-19,-20,-21,-22,-23,-24,-25,-26,-27,-28,-29,-30,-31,-32,-33,-34,-35])
            if qvcenter == 2.213:
                W2edges = np.delete(W2edges,[16,17])
            W2centers = (W2edges[:-1] + W2edges[1:]) / 2
            df.loc[mask, 'W2bin_qv'] = pd.cut(df.loc[mask, 'W2'], bins = W2edges, labels = False, include_lowest = True)
            df.loc[mask, 'W2center_qv'] = df.loc[mask, 'W2bin_qv'].map(lambda i: W2centers[int(i)] if pd.notnull(i) else np.nan)
            df.loc[mask, 'nucenter_w2_qv'] = np.sqrt(qvcenter**2 + df.loc[mask, 'W2center_qv']) - mass_nucleon
            df.loc[mask, 'epcenter_w2_qv'] = 1 / (1 + 2 * (1 + (df.loc[mask, 'nucenter_w2_qv']**2) / (qvcenter**2 - df.loc[mask, 'nucenter_w2_qv']**2)) * df.loc[mask, 'tan2(T/2)']) 

    W2ns = np.array([10,10,10,10,25,
                25,20,15,15,15,
                15,15,15,15,15,
                15,15,15])

    for i in range(len(Q2centers)):
        Q2center = Q2centers[i]

        # Ex < 30MeV:
        mask = (df['Q2center'] == Q2center) & (df['Ex'] < Ex_cut)
        if Q2center == 0.01:
            mask = (df['Q2center'] == Q2center) & (df['Ex'] < Ex_cut_lowq)
        if len(df.loc[mask, 'Ex']) > 0:
            n_bins = max(1,len(df.loc[mask, 'Ex']) // 5)
            Exedges = np.quantile(df.loc[mask, 'Ex'], np.linspace(0, 1, n_bins + 1))
            Exedges = Exedges_epsilon_range(df = df.loc[mask], edges = Exedges)
            Excenters = (Exedges[:-1] + Exedges[1:]) / 2
            df.loc[mask, 'Exbin_q2'] = pd.cut(df.loc[mask, 'Ex'], bins=Exedges, labels=False, include_lowest=True,duplicates='drop')
            df.loc[mask, 'Excenter_q2'] = df.loc[mask, 'Exbin_q2'].map(lambda i: Excenters[int(i)] if pd.notnull(i) else np.nan)
            df.loc[mask, 'nucenter_ex_q2'] = df.loc[mask, 'Excenter_q2'] + Q2center / (2 * mass_nucleus)
            df.loc[mask, 'epcenter_ex_q2'] = 1 / (1 + 2 * (1 + (df.loc[mask, 'nucenter_ex_q2']**2) / Q2center) * df.loc[mask, 'tan2(T/2)']) 

        # Ex >= 30MeV:
        mask = (df['Q2center'] == Q2center) & (df['Ex'] >= Ex_cut)
        if Q2center == 0.01:
            mask = (df['Q2center'] == Q2center) & (df['Ex'] > Ex_cut_lowq)
        if len(df.loc[mask, 'W2']) > 0:
            n_bins = max(1, len(df.loc[mask, 'W2']) // W2ns[i])
            W2edges = np.quantile(df.loc[mask, 'W2'], np.linspace(0, 1, n_bins + 1))
            if Q2center == 0.02: 
                W2edges = np.delete(W2edges, [-12,-13,-14,-15,-17,-18])
            if Q2center == 0.026:
                W2edges = np.delete(W2edges, [-2,-3,-4,-8])
            if Q2center == 0.04:
                W2edges = np.delete(W2edges, [-3,-4,-5,-6,-8,-9])
            if Q2center == 0.056:
                W2edges = np.delete(W2edges, [-3,-4,-7,-8])
            if Q2center == 0.093:
                W2edges = np.delete(W2edges, [-4,-20,-22])
            if Q2center == 0.12:
                W2edges = np.delete(W2edges, [-3,-4,-5,-6])
            if Q2center == 0.16:
                W2edges = np.delete(W2edges, [1,7,-4,-5])
            if Q2center == 0.38:
                W2edges = np.delete(W2edges, [-2,-3,-4,-7,-8,-10,-11])
            if Q2center == 2.25:
                W2edges = np.delete(W2edges, [16])
            W2centers = (W2edges[:-1] + W2edges[1:]) / 2
            df.loc[mask, 'W2bin_q2'] = pd.cut(df.loc[mask, 'W2'], bins = W2edges, labels = False, include_lowest = True, duplicates = 'drop')
            df.loc[mask, 'W2center_q2'] = df.loc[mask, 'W2bin_q2'].map(lambda i: W2centers[int(i)] if pd.notnull(i) else np.nan)
            df.loc[mask, 'nucenter_w2_q2'] = (df.loc[mask, 'W2center_q2'] - mass_nucleon**2 + Q2center) / (2 * mass_nucleon)
            df.loc[mask, 'epcenter_w2_q2'] = 1 / (1 + 2 * (1 + (df.loc[mask, 'nucenter_w2_q2']**2) / Q2center) * df.loc[mask, 'tan2(T/2)']) 

    df = df.dropna()
    return df

In [5]:
def calculate_bc(df):
    response_columns = ['i','RTTOT','RLTOT','RTnoNS','RLnoNS']

    # calculate bc_qv_ex
    df[['qvcenter','Excenter_qv']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_qv_ex.exe', 'input.txt'], stdout=output_file) 
    time.sleep(0.5)
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_qvc_ex'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_qvc_ex'] = df.index.map(df_response.set_index('i')['RTTOT'])
    df['RT_qvc_ex'] = df['RT_qvc_ex'] + RT_quasi_deuteron(nus = df['nucenter_ex_qv'], q2s = df['qvcenter']**2 - df['nucenter_ex_qv']**2, exs = df['Excenter_qv'])

    df[['qv','Ex']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_qv_ex.exe', 'input.txt'], stdout=output_file) 
    time.sleep(0.5)
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_qvd_ex'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_qvd_ex'] = df.index.map(df_response.set_index('i')['RTTOT'])
    df['RT_qvd_ex'] = df['RT_qvd_ex'] + df['RT_QD_data']

    df['bc_qv_ex'] = 1.0
    for qvcenter in qvcenters:
        # Ex < 30MeV:
        mask = (df['qvcenter'] == qvcenter) & (df['Ex'] < Ex_cut)
        if qvcenter == 0.1:
            mask = (df['qvcenter'] == qvcenter) & (df['Ex'] < Ex_cut_lowq)
        df.loc[mask, 'bc_qv_ex'] = (df.loc[mask, 'epcenter_ex_qv'] * df.loc[mask, 'RL_qvc_ex'] + 0.5 * ((qvcenter**2) / (qvcenter**2 - df.loc[mask,'nucenter_ex_qv']**2)) * df.loc[mask, 'RT_qvc_ex']) / (df.loc[mask, 'epsilon'] * df.loc[mask, 'RL_qvd_ex'] + 0.5 * (df.loc[mask, 'qv2'] / df.loc[mask, 'Q2']) * df.loc[mask, 'RT_qvd_ex'])
    print('RL RT bc_qv_ex done.')

    # calculate bc_qv_w2
    df[['qvcenter','W2center_qv']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_qv_w2.exe', 'input.txt'], stdout=output_file) 
    time.sleep(0.5)
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_qvc_w2'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_qvc_w2'] = df.index.map(df_response.set_index('i')['RTTOT'])
    df['RT_qvc_w2'] = df['RT_qvc_w2'] + RT_quasi_deuteron(nus = df['nucenter_w2_qv'], q2s = df['qvcenter']**2 - df['nucenter_w2_qv']**2, exs = df['nucenter_w2_qv'] - (df['qvcenter']**2 - df['nucenter_w2_qv']**2) / (2 * mass_nucleus))

    df[['qv','W2']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_qv_w2.exe', 'input.txt'], stdout=output_file) 
    time.sleep(0.5)
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_qvd_w2'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_qvd_w2'] = df.index.map(df_response.set_index('i')['RTTOT'])
    df['RT_qvd_w2'] = df['RT_qvd_w2'] + df['RT_QD_data']
    

    df['bc_qv_w2'] = 1.0
    for qvcenter in qvcenters:
        # Ex >= 30MeV:
        mask = (df['qvcenter'] == qvcenter) & (df['Ex'] >= Ex_cut)
        df.loc[mask, 'bc_qv_w2'] = (df.loc[mask, 'epcenter_w2_qv'] * df.loc[mask, 'RL_qvc_w2'] + 0.5 * ((qvcenter**2) / (qvcenter**2 - df.loc[mask,'nucenter_w2_qv']**2)) * df.loc[mask, 'RT_qvc_w2']) / (df.loc[mask, 'epsilon'] * df.loc[mask, 'RL_qvd_w2'] + 0.5 * (df.loc[mask, 'qv2'] / df.loc[mask, 'Q2']) * df.loc[mask, 'RT_qvd_w2'])
    print('RL RT bc_qv_w2 done.')

    # calculate bc_q2_ex
    df[['Q2center','Excenter_q2']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_q2_ex.exe', 'input.txt'], stdout=output_file) 
    time.sleep(0.5)
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_q2c_ex'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_q2c_ex'] = df.index.map(df_response.set_index('i')['RTTOT'])
    df['RT_q2c_ex'] = df['RT_q2c_ex'] + RT_quasi_deuteron(nus = df['nucenter_ex_q2'], q2s = df['Q2center'], exs = df['Excenter_q2'])

    df[['Q2','Ex']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_q2_ex.exe', 'input.txt'], stdout=output_file) 
    time.sleep(0.5)
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_q2d_ex'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_q2d_ex'] = df.index.map(df_response.set_index('i')['RTTOT'])
    df['RT_q2d_ex'] = df['RT_q2d_ex'] + df['RT_QD_data']

    df['bc_q2_ex'] = 1.0
    for Q2center in Q2centers:
        # Ex < 30MeV:
        mask = (df['Q2center'] == Q2center) & (df['Ex'] < Ex_cut)
        df.loc[mask, 'bc_q2_ex'] = (df.loc[mask, 'epcenter_ex_q2'] * df.loc[mask, 'RL_q2c_ex'] + 0.5 * ((Q2center + df.loc[mask, 'nucenter_ex_q2']**2) / Q2center) * df.loc[mask, 'RT_q2c_ex']) / (df.loc[mask, 'epsilon'] * df.loc[mask, 'RL_q2d_ex'] + 0.5 * (df.loc[mask, 'qv2'] / df.loc[mask, 'Q2']) * df.loc[mask, 'RT_q2d_ex'])
    print('RL RT bc_q2_ex done.')

    # calculate bc_q2_w2
    df[['Q2center','W2center_q2']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_q2_w2.exe', 'input.txt'], stdout=output_file) 
    time.sleep(0.5)
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_q2c_w2'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_q2c_w2'] = df.index.map(df_response.set_index('i')['RTTOT'])
    df['RT_q2c_w2'] = df['RT_q2c_w2'] + RT_quasi_deuteron(nus = df['nucenter_w2_q2'], q2s = df['Q2center'], exs = df['nucenter_w2_q2'] - df['Q2center'] / (2 * mass_nucleus))

    df[['Q2','W2']].to_csv('input.txt',index=True,header=False,sep=' ')
    with open('output.txt', 'w') as output_file:
        subprocess.run(['response/response_q2_w2.exe', 'input.txt'], stdout=output_file) 
    time.sleep(0.5)
    df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
    df['RL_q2d_w2'] = df.index.map(df_response.set_index('i')['RLTOT'])
    df['RT_q2d_w2'] = df.index.map(df_response.set_index('i')['RTTOT'])
    df['RT_q2d_w2'] = df['RT_q2d_w2'] + df['RT_QD_data']

    df['bc_q2_w2'] = 1.0
    for Q2center in Q2centers:
        # Ex >= 30MeV:
        mask = (df['Q2center'] == Q2center) & (df['Ex'] >= Ex_cut)
        df.loc[mask, 'bc_q2_w2'] = (df.loc[mask, 'epcenter_w2_q2'] * df.loc[mask, 'RL_q2c_w2'] + 0.5 * ((Q2center + df.loc[mask, 'nucenter_w2_q2']**2) / Q2center) * df.loc[mask, 'RT_q2c_w2']) / (df.loc[mask, 'epsilon'] * df.loc[mask, 'RL_q2d_w2'] + 0.5 * (df.loc[mask, 'qv2'] / df.loc[mask, 'Q2']) * df.loc[mask, 'RT_q2d_w2'])
    print('RL RT bc_q2_w2 done.')

    return df

<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:18: SyntaxWarning: invalid escape sequence '\s'
<>:37: SyntaxWarning: invalid escape sequence '\s'
<>:46: SyntaxWarning: invalid escape sequence '\s'
<>:64: SyntaxWarning: invalid escape sequence '\s'
<>:73: SyntaxWarning: invalid escape sequence '\s'
<>:90: SyntaxWarning: invalid escape sequence '\s'
<>:99: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:18: SyntaxWarning: invalid escape sequence '\s'
<>:37: SyntaxWarning: invalid escape sequence '\s'
<>:46: SyntaxWarning: invalid escape sequence '\s'
<>:64: SyntaxWarning: invalid escape sequence '\s'
<>:73: SyntaxWarning: invalid escape sequence '\s'
<>:90: SyntaxWarning: invalid escape sequence '\s'
<>:99: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Rhys\AppData\Local\Temp\ipykernel_21116\529503794.py:9: SyntaxWarning: invalid escape sequence '\s'
  df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=respo

In [6]:
df = pd.read_csv('Data/C12.csv')
df = df.loc[df['dataSet'] != 6]
df = df.loc[df['dataSet'] != 25]
df = prepare_df(df)
df = calculate_bc(df)
df.to_csv('Data/df_C12.csv',index=False)
df

c:\Users\Rhys\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\Rhys\AppData\Local\Temp\ipykernel_21116\2549002893.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Exbin_qv'] = 0.0
C:\Users\Rhys\AppData\Local\Temp\ipykernel_21116\2549002893.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Excenter_qv'] = 0.0
C:\Users\Rhys\AppData\Local\Temp\ipykernel

[18]
RL RT bc_qv_ex done.
RL RT bc_qv_w2 done.
RL RT bc_q2_ex done.
RL RT bc_q2_w2 done.


,Z,A,E0,ThetaDeg,nu,cross,error,dataSet,normalization,normError,...,RL_q2c_ex,RT_q2c_ex,RL_q2d_ex,RT_q2d_ex,bc_q2_ex,RL_q2c_w2,RT_q2c_w2,RL_q2d_w2,RT_q2d_w2,bc_q2_w2
0,6,12,0.1227,90.0,0.0071,86760.0,9860.0,1,0.9919,0.0024,...,0.044642,1.099419e-04,0.070970,3.039044e-06,0.630537,0.000000,0.000000,0.070984,3.026265e-06,1.000000
1,6,12,0.1227,90.0,0.0121,26350.0,5890.0,1,0.9919,0.0024,...,0.044642,1.099419e-04,0.029609,9.139163e-04,1.447471,0.000000,0.000000,0.029286,9.507651e-04,1.000000
2,6,12,0.1227,90.0,0.0171,25810.0,4670.0,1,0.9919,0.0024,...,0.019967,7.951140e-03,0.020010,8.048238e-03,0.994084,0.000000,0.000000,0.020097,8.191040e-03,1.000000
3,6,12,0.1227,90.0,0.0221,40060.0,4770.0,1,0.9919,0.0024,...,0.033580,1.205698e-02,0.033214,1.342523e-02,0.967290,0.000000,0.000000,0.033861,1.352629e-02,1.000000
4,6,12,0.1227,145.0,0.0021,17100.0,700.0,1,0.9919,0.0024,...,0.060424,2.389449e-09,0.013657,2.990983e-11,4.422814,0.000000,0.000000,0.012407,2.623948e-11,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9925,6,12,0.8581,70.0,0.5255,599.7,9.5,24,1.0300,0.0200,...,0.000003,8.728022e-14,0.002736,1.550036e-02,1.000000,0.003347,0.016499,0.002872,1.615117e-02,1.160875
9926,6,12,0.8581,70.0,0.5355,636.8,17.5,24,1.0300,0.0200,...,0.000003,8.728022e-14,0.002901,1.639444e-02,1.000000,0.003347,0.016499,0.003010,1.692335e-02,1.079271
9927,6,12,0.8581,70.0,0.5455,644.1,18.9,24,1.0300,0.0200,...,0.000003,8.728022e-14,0.003029,1.724011e-02,1.000000,0.003347,0.016499,0.003151,1.760172e-02,1.009193
9928,6,12,0.8581,70.0,0.5550,626.6,18.9,24,1.0300,0.0200,...,0.000003,8.728022e-14,0.003158,1.796071e-02,1.000000,0.003347,0.016499,0.003286,1.814485e-02,0.951950
